# Mutual Fund Analysis Workspace

This notebook loads the provided CSV datasets from `data/raw`, fetches live NAV data from mfapi.in, and performs the AMFI code validation checks requested for the project. If the CSV inputs are not present in the workspace, the relevant cells will report that cleanly instead of failing.

In [1]:
from pathlib import Path
import json
import pandas as pd
import requests

def find_project_root(start: Path | None = None) -> Path:
    current = start or Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'raw').exists() and (candidate / 'notebooks').exists():
            return candidate
    return current

project_root = find_project_root()
raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print(f'Project root: {project_root}')
print(f'Raw data folder: {raw_dir}')

Project root: c:\repos\cp1_Mutual_Fund_Analysis
Raw data folder: c:\repos\cp1_Mutual_Fund_Analysis\data\raw


In [2]:
csv_files = sorted(raw_dir.glob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) in data/raw.')

loaded_frames = {}
for csv_path in csv_files:
    print(f'\n=== {csv_path.name} ===')
    frame = pd.read_csv(csv_path)
    loaded_frames[csv_path.stem] = frame
    print('shape:', frame.shape)
    print('dtypes:')
    print(frame.dtypes)
    print('head:')
    print(frame.head())

    anomalies = []
    if frame.columns.duplicated().any():
        anomalies.append('duplicate column names')
    unnamed_columns = [col for col in frame.columns if str(col).startswith('Unnamed')]
    if unnamed_columns:
        anomalies.append(f'unnamed columns: {unnamed_columns}')
    duplicate_rows = int(frame.duplicated().sum())
    if duplicate_rows:
        anomalies.append(f'{duplicate_rows} duplicated row(s)')
    missing_counts = frame.isna().sum().sort_values(ascending=False)
    missing_counts = missing_counts[missing_counts > 0]
    if not missing_counts.empty:
        anomalies.append(f'missing values in {list(missing_counts.index)}')
    if anomalies:
        print('anomalies:', '; '.join(anomalies))
    else:
        print('anomalies: none detected by the basic checks')

if not csv_files:
    print('No CSV datasets were available in data/raw, so the dataset profiling step could not run yet.')

Found 10 CSV file(s) in data/raw.

=== 01_fund_master.csv ===
shape: (40, 15)
dtypes:
amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object
head:
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fu

In [3]:
def fetch_nav_history(scheme_code: int, scheme_label: str) -> pd.DataFrame:
    url = f'https://api.mfapi.in/mf/{scheme_code}'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    payload = response.json()

    nav_rows = payload.get('data', [])
    frame = pd.DataFrame(nav_rows)
    if not frame.empty:
        rename_map = {}
        for candidate in ['date', 'nav', 'scheme_name']:
            if candidate not in frame.columns:
                continue
        frame['scheme_code'] = scheme_code
        frame['scheme_label'] = scheme_label
        if 'date' in frame.columns:
            frame['date'] = pd.to_datetime(frame['date'], errors='coerce', dayfirst=True)
        if 'nav' in frame.columns:
            frame['nav'] = pd.to_numeric(frame['nav'], errors='coerce')
        frame = frame[[col for col in ['scheme_code', 'scheme_label', 'date', 'nav'] if col in frame.columns] + [col for col in frame.columns if col not in {'scheme_code', 'scheme_label', 'date', 'nav'}]]

    output_path = raw_dir / f'nav_{scheme_code}.csv'
    frame.to_csv(output_path, index=False)
    print(f'Saved {output_path.name} with shape {frame.shape}')
    return frame

primary_nav = fetch_nav_history(125497, 'HDFC Top 100 Direct')
key_schemes = [
    (119551, 'SBI Bluechip'),
    (120503, 'ICICI Bluechip'),
    (118632, 'Nippon Large Cap'),
    (119092, 'Axis Bluechip'),
    (120841, 'Kotak Bluechip'),
]

fetched_frames = {'125497': primary_nav}
for scheme_code, scheme_label in key_schemes:
    fetched_frames[str(scheme_code)] = fetch_nav_history(scheme_code, scheme_label)

Saved nav_125497.csv with shape (3092, 4)
Saved nav_119551.csv with shape (3237, 4)
Saved nav_120503.csv with shape (3308, 4)
Saved nav_118632.csv with shape (3299, 4)
Saved nav_119092.csv with shape (3566, 4)
Saved nav_120841.csv with shape (3302, 4)


In [4]:
fund_master_frame = None
fund_master_candidates = [path for path in csv_files if 'fund' in path.name.lower() and 'master' in path.name.lower()]
if fund_master_candidates:
    fund_master_path = fund_master_candidates[0]
    fund_master_frame = pd.read_csv(fund_master_path)
    print(f'Loaded fund master: {fund_master_path.name} with shape {fund_master_frame.shape}')

    for column in ['fund_house', 'fund house', 'category', 'sub_category', 'sub-category', 'risk', 'risk_grade', 'risk grade']:
        matching = [col for col in fund_master_frame.columns if col.lower() == column.replace(' ', '_').replace('-', '_')]
        if matching:
            print(f'Unique values for {matching[0]}:')
            print(sorted(fund_master_frame[matching[0]].dropna().astype(str).unique().tolist())[:50])

    possible_code_columns = [col for col in fund_master_frame.columns if any(token in col.lower() for token in ['amfi', 'scheme_code', 'scheme code', 'code'])]
    print('Possible AMFI code columns:', possible_code_columns)
else:
    print('fund_master dataset not found in data/raw, so fund-house/category/risk exploration is pending.')

nav_history_frame = None
nav_candidates = [path for path in csv_files if 'nav' in path.name.lower() and 'history' in path.name.lower()]
if nav_candidates:
    nav_history_path = nav_candidates[0]
    nav_history_frame = pd.read_csv(nav_history_path)
    print(f'Loaded nav_history: {nav_history_path.name} with shape {nav_history_frame.shape}')

    def pick_code_column(frame: pd.DataFrame) -> str | None:
        candidates = [col for col in frame.columns if any(token in col.lower() for token in ['amfi', 'scheme_code', 'scheme code', 'code'])]
        return candidates[0] if candidates else None

    fund_master_code_col = pick_code_column(fund_master_frame) if fund_master_frame is not None else None
    nav_history_code_col = pick_code_column(nav_history_frame)

    if fund_master_frame is not None and fund_master_code_col and nav_history_code_col:
        fund_codes = set(pd.to_numeric(fund_master_frame[fund_master_code_col], errors='coerce').dropna().astype(int).tolist())
        nav_codes = set(pd.to_numeric(nav_history_frame[nav_history_code_col], errors='coerce').dropna().astype(int).tolist())
        missing_codes = sorted(fund_codes - nav_codes)
        print(f'AMFI code count in fund_master: {len(fund_codes)}')
        print(f'AMFI code count in nav_history: {len(nav_codes)}')
        print(f'Codes in fund_master but not nav_history: {len(missing_codes)}')
        print('Sample missing codes:', missing_codes[:20])
    else:
        print('AMFI validation could not run because the expected code columns were not found.')
else:
    print('nav_history dataset not found in data/raw, so AMFI validation is pending.')

quality_summary = {
    'csv_files_found': len(csv_files),
    'loaded_frames': list(loaded_frames.keys()),
    'fetched_nav_files': [f'nav_{code}.csv' for code in [125497, 119551, 120503, 118632, 119092, 120841]],
    'validation_status': 'complete only when fund_master and nav_history datasets are available'
}
print(json.dumps(quality_summary, indent=2, default=str))

Loaded fund master: 01_fund_master.csv with shape (40, 15)
Unique values for fund_house:
['Aditya Birla Sun Life MF', 'Axis Mutual Fund', 'DSP Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Kotak Mahindra MF', 'Mirae Asset MF', 'Nippon India MF', 'SBI Mutual Fund', 'UTI Mutual Fund']
Unique values for fund_house:
['Aditya Birla Sun Life MF', 'Axis Mutual Fund', 'DSP Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Kotak Mahindra MF', 'Mirae Asset MF', 'Nippon India MF', 'SBI Mutual Fund', 'UTI Mutual Fund']
Unique values for category:
['Debt', 'Equity']
Unique values for sub_category:
['ELSS', 'Flexi Cap', 'Gilt', 'Index', 'Index/ETF', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Short Duration', 'Small Cap', 'Value']
Unique values for sub_category:
['ELSS', 'Flexi Cap', 'Gilt', 'Index', 'Index/ETF', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Short Duration', 'Small Cap', 'Value']
Possible AMFI code columns: ['amfi_code', 'sebi_category_code']
Lo

In [5]:
frames = globals().get('loaded_frames', {})
if not frames:
    frames = {path.stem: pd.read_csv(path) for path in sorted(raw_dir.glob('*.csv'))}

summary_rows = []
for name, frame in frames.items():
    missing_cells = int(frame.isna().sum().sum())
    duplicate_rows = int(frame.duplicated().sum())
    duplicate_columns = [col for col in frame.columns[frame.columns.duplicated()]]
    unnamed_columns = [col for col in frame.columns if str(col).startswith('Unnamed')]
    summary_rows.append({
        'dataset': name,
        'shape': frame.shape,
        'missing_cells': missing_cells,
        'duplicate_rows': duplicate_rows,
        'duplicate_columns': duplicate_columns or None,
        'unnamed_columns': unnamed_columns or None,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('dataset')
print(summary_df.to_string(index=False))

anomaly_notes = []
for _, row in summary_df.iterrows():
    notes = []
    if row['missing_cells']:
        notes.append(f"{row['missing_cells']} missing cell(s)")
    if row['duplicate_rows']:
        notes.append(f"{row['duplicate_rows']} duplicate row(s)")
    if row['duplicate_columns']:
        notes.append('duplicate column names')
    if row['unnamed_columns']:
        notes.append('unnamed columns present')
    if notes:
        anomaly_notes.append(f"{row['dataset']}: {', '.join(notes)}")

if anomaly_notes:
    print('Anomaly notes:')
    for note in anomaly_notes:
        print(f'- {note}')
else:
    print('Anomaly notes: none detected by the basic checks')

                 dataset       shape  missing_cells  duplicate_rows duplicate_columns unnamed_columns
          01_fund_master    (40, 15)              0               0              None            None
          02_nav_history  (46000, 3)              0               0              None            None
    03_aum_by_fund_house     (90, 5)              0               0              None            None
  04_monthly_sip_inflows     (48, 6)             12               0              None            None
     05_category_inflows    (144, 3)              0               0              None            None
 06_industry_folio_count     (21, 6)              0               0              None            None
   07_scheme_performance    (40, 19)              0               0              None            None
08_investor_transactions (32778, 13)              0               0              None            None
   09_portfolio_holdings    (322, 8)              0               0              N

## Notes

- The live NAV pulls will save raw CSVs under `data/raw`.
- The dataset profiling and AMFI validation cells will complete once the ten provided CSV datasets are placed in `data/raw`.
- If the raw inputs are added later, rerun the notebook from top to bottom to produce the final data-quality summary.